# Create & Populate QUAM State — OPX+ / Octave

Standalone notebook for initial hardware bring-up.

**Steps**:
1. [Preamble](#0-preamble) — switch to the correct Qualibrate project
2. [Create QUAM state](#1-create-quam-state) — allocate wiring and build `state.json` / `wiring.json` *(run once)*
3. [Populate QUAM with initial values](#2-populate-quam-with-initial-values) — write all hardware parameters into the state *(re-run to reset)*
4. [Verify state](#3-verify-state) — sanity-check the saved structure

**Hardware layout** (OPX+ con1 + Octave oct1):

| Channel       | OPX+ ports (I/Q) | Digital | Octave RF out | LO source |
|---------------|------------------|---------|---------------|-----------|
| Resonator     | 1 / 2            | 1       | RF_outputs/1  | internal synth1 |
| f0g1 sideband | 3 / 4            | 3       | RF_outputs/2  | **external** LO2 |
| Qubit XY      | 5 / 6            | 5       | RF_outputs/3  | internal synth3 |
| Cavity        | 7 / 8            | 7       | RF_outputs/4  | internal synth4 |

## 0. Preamble

Run this cell first every session to select the correct Qualibrate project.

In [ ]:
import sys, os
# Add this notebook's directory to sys.path so the local quam_config package
# is found before any installed version.
_here = os.path.abspath('')
if _here not in sys.path:
    sys.path.insert(0, _here)
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path
from qualibrate_config.core.project.switch import switch_project

config_path = get_qualibrate_config_path()
config = get_qualibrate_config(config_path)
print(f"Current project: {config.project}")

desired_project = "automatic_calibration"  # <-- edit if needed
if config.project != desired_project:
    switch_project(config_path, desired_project)
    config = get_qualibrate_config(config_path)
    print(f"Switched to project: {config.project}")
else:
    print(f"Project already set to '{desired_project}'")

print(f"Storage location: {config.storage.location}")

## 1. Create QUAM state

Run **once** to build `state.json` and `wiring.json` in `quam_state/`.
Skip (or re-run to regenerate wiring) if the files already exist.

> The f0g1 and cavity channels are **not** allocated here — they are added
> manually in the populate cell below because they share OPX+ ports with
> specific Octave RF outputs that the wirer does not auto-assign.

In [ ]:
import matplotlib.pyplot as plt
from qualang_tools.wirer.wirer.channel_specs import (
    octave_spec, opx_iq_octave_spec, opx_dig_spec, ChannelSpecOctaveDigital
)
from qualang_tools.wirer import Instruments, Connectivity, allocate_wiring, visualize
from quam_builder.builder.qop_connectivity import build_quam_wiring
from quam_builder.builder.superconducting import build_quam
from quam.components.channels import DigitalOutputChannel
from quam_config import Quam

########################################################################################################################
# %%  Static parameters
########################################################################################################################
host_ip             = "192.168.3.50"  # OPX+ host IP
port                = None
cluster_name        = "Cluster_1"
calibration_db_path = None

########################################################################################################################
# %%  Instruments
########################################################################################################################
instruments = Instruments()
instruments.add_opx_plus(controllers=[1])
instruments.add_octave(indices=1)

qubits = [1]

########################################################################################################################
# %%  Channel addresses
########################################################################################################################
# Resonator  -> RF_outputs/1 (int LO synth1), OPX+ ports 1(I)/2(Q), trigger 1
# Qubit XY   -> RF_outputs/3 (int LO synth3), OPX+ ports 5(I)/6(Q), trigger 5
# f0g1 / cavity channels are added in the populate cell (shared/manual ports).
qubit_res_ch = (
    opx_iq_octave_spec(
        con=1,
        out_port_i=1, out_port_q=2,
        in_port_i=1,  in_port_q=2,
        octave_index=1, rf_out=1, rf_in=1,
    )
    & opx_dig_spec(con=1, out_port=1)
    & ChannelSpecOctaveDigital(con=1, in_port=1)
)
qubit_xy_ch = (
    opx_iq_octave_spec(
        con=1,
        out_port_i=5, out_port_q=6,
        octave_index=1, rf_out=3,
    )
    & opx_dig_spec(con=1, out_port=5)
    & ChannelSpecOctaveDigital(con=1, in_port=3)
)

########################################################################################################################
# %%  Allocate wiring
########################################################################################################################
connectivity = Connectivity()
connectivity.add_resonator_line(qubits=qubits, triggered=True, constraints=qubit_res_ch)
connectivity.add_qubit_drive_lines(qubits=qubits, triggered=True, constraints=qubit_xy_ch)
allocate_wiring(connectivity, instruments)

fig_wiring = visualize(connectivity.elements, available_channels=instruments.available_channels)
plt.show(block=False)

########################################################################################################################
# %%  Build and save
########################################################################################################################
user_input = input("Save QUAM? (y/n) ").strip().lower()
if user_input == "y":
    machine = Quam()
    build_quam_wiring(connectivity, host_ip, cluster_name, machine)
    machine = Quam.load()
    build_quam(machine, calibration_db_path)

    # All active RF outputs use internal LO — set triggered output mode
    for octave in machine.octaves.values():
        for rf_out in octave.RF_outputs.values():
            if rf_out.channel is not None:
                rf_out.output_mode = "triggered"

    machine.save()
    print("QUAM state created and saved.")
    print("Next: run the populate cell to write all hardware parameters.")
else:
    print("Skipped — no changes saved.")

## 2. Populate QUAM with initial values

Edit the **USER PARAMETERS** section to match your chip specs, then run.

Re-run at any time to reset the state back to the starting-point values.
Calibration nodes overwrite specific fields; this cell resets everything.

**Port map reminder**:
- RF_outputs/1 (int LO) — resonator, OPX+ 1/2
- RF_outputs/2 (ext LO) — f0g1 sideband, OPX+ 3/4 ← added here
- RF_outputs/3 (int LO) — qubit XY, OPX+ 5/6
- RF_outputs/4 (int LO) — cavity (alice + bob shared), OPX+ 7/8 ← added here

In [ ]:
import json
import numpy as np
from pprint import pprint
from qualang_tools.units import unit
from quam.components.pulses import SquarePulse, DragCosinePulse, DragGaussianPulse
from quam.components.octave import OctaveUpConverter
from quam.components.channels import DigitalOutputChannel
from quam.components.ports import (
    OPXPlusAnalogInputPort, OPXPlusAnalogOutputPort, OPXPlusDigitalOutputPort
)
from quam_builder.architecture.superconducting.components.xy_drive import XYDriveIQ
from quam_builder.architecture.superconducting.cavity.cavity import Cavity
from quam_builder.architecture.superconducting.cavity.cavity_mode import CavityMode
from quam_builder.builder.superconducting.pulses import add_DragGaussian_pulses
from quam_config import Quam, TemporaryCalibrationData
from quam_builder.architecture.superconducting.qubit_pair import CavityTransmonPair

u = unit(coerce_to_integer=True)

########################################################################################################################
# %%  Helper
########################################################################################################################
def get_octave_gain_and_amplitude(desired_power: float, max_amplitude: float = 0.125):
    """Convert desired output power (dBm) to Octave gain + OPX+ IF amplitude.

    Octave gain range: -20 to +20 dB in 0.5 dB steps.
    OPX+ IF amplitude: < 0.5 V (practical limit ~0.125 V for good linearity).
    """
    octave_gain = round(max(min(desired_power - u.volts2dBm(max_amplitude), 20), -20) * 2) / 2
    amplitude   = u.dBm2volts(desired_power - octave_gain)
    if not (-20 <= octave_gain <= 20 and -0.5 <= amplitude < 0.5):
        raise ValueError(f"Power outside spec: gain={octave_gain}, amp={amplitude:.4f}")
    return octave_gain, amplitude


########################################################################################################################
# %%  Load machine
########################################################################################################################
machine = Quam.load()

########################################################################################################################
# %%  USER PARAMETERS -- edit to match chip specs
########################################################################################################################
CAVITY_ID = "c1"   # key used in machine.cavities

# -- Readout resonator (RF_outputs/1, OPX+ 1/2, trigger 1) -----------------------
rr_freq             = 7.500e9   # Hz  dressed resonator frequency
rr_LO               = 7.400e9   # Hz  Octave RF_outputs/1 internal LO
readout_power       = -10        # dBm output power at cavity input
readout_gain        = -10        # dB  gain for input readout amplifiers
readout_adc_gain_db = 0          # dB  OPX+ ADC input gain [-12, +20], 0 dB nominal

# -- Qubit XY drive (RF_outputs/3, OPX+ 5/6, trigger 5) -------------------------
xy_freq       = 4.600e9   # Hz  qubit ge transition frequency
xy_LO         = 4.400e9   # Hz  Octave RF_outputs/3 internal LO
anharmonicity = -200e6    # Hz  transmon anharmonicity (negative)
drive_power   = -10        # dBm qubit drive power

# -- Cavity drives (RF_outputs/4, OPX+ 7/8, trigger 7) --------------------------
# Alice and Bob share the same RF output and LO.
alice_freq  = 6.000e9   # Hz  Alice cavity mode frequency
alice_LO    = 5.900e9   # Hz  Octave RF_outputs/4 internal LO (shared with Bob)
alice_power = 20         # dBm cavity drive power
bob_freq    = 6.200e9   # Hz  Bob cavity mode frequency
bob_power   = 20         # dBm cavity drive power (same RF output as Alice)

# -- f0g1 sideband drive (RF_outputs/2, ext LO, OPX+ 3/4, trigger 3) ------------
alice_f0g1_freq                 = 3.25e9   # Hz  initial estimate; refine after node 21
alice_f0g1_LO                   = 3.0e9    # Hz  external LO connected to Octave LO2 input
alice_f0g1_gain                 = 0         # dB  Octave RF_outputs/2 gain [-20, +20]
alice_f0g1_saturation_length_ns = 20000    # ns  long square pulse for spectroscopy
alice_f0g1_pi_length_ns         = 1000     # ns  Gaussian pi pulse length
alice_f0g1_sigma_ns             = 200      # ns  Gaussian sigma (typically length/5)
alice_f0g1_amp                  = 0.4      # V   initial pulse amplitude

# -- Resonator bare frequency and timing -----------------------------------------
rr_freq_bare      = 7.504e9   # Hz  bare resonator frequency (before dispersive shift)
tof_ns            = 224        # ns  time-of-flight (refine with node 01a_time_of_flight)

# -- Qubit reset / timing --------------------------------------------------------
thermalization_time_factor = 5   # x T1 wait for qubit thermal reset
sigma_time_factor          = 5

# -- EF pi-pulse (initial values; calibrated by nodes 12 / 13) -------------------
ef_x180_length_ns  = 40    # ns  same length as ge x180
ef_x180_amplitude  = 0.1   # V   rough initial amplitude
ef_x180_sigma_ns   = 8     # ns  Gaussian sigma

# -- Selective pi-pulse (narrow-bandwidth; calibrated by node 04b_power_rabi) ----
selective_x180_length_ns = 10000  # ns  10 us -> ~100 kHz bandwidth

# -- Pulse defaults --------------------------------------------------------------
readout_length_ns                = 8000
saturation_length_ns             = 20000
x180_length_ns                   = 1000
gaussian_sigma_ns                = x180_length_ns // 5   # 200 ns for 1 us pulse
drag_alpha                       = 0.0
drag_detuning                    = 0.0
displacement_length_ns           = 1000
displacement_sigma_ns            = displacement_length_ns // 5
displacement_initial_amplitude_V = 0.001   # V  (adjust if peak outside sweep range)

# -- T1 estimates ----------------------------------------------------------------
T1                          = 200e-6   # seconds  transmon
cavity_T1                   = 100e-6   # seconds  cavity modes
resonator_depletion_time_ns = 10000    # ns

########################################################################################################################
# %%  Assertions
########################################################################################################################
assert abs(rr_freq - rr_LO)                  < 400e6, "Resonator IF out of range"
assert abs(xy_freq - xy_LO)                  < 400e6, "XY IF out of range"
assert abs(alice_freq - alice_LO)            < 400e6, "Alice IF out of range"
assert abs(bob_freq   - alice_LO)            < 400e6, "Bob IF out of range (must share LO with Alice)"
assert abs(alice_f0g1_freq - alice_f0g1_LO)  < 400e6, "f0g1 IF out of Octave range (must be <400 MHz)"
print("All frequency assertions passed.")

########################################################################################################################
# %%  Resonator hardware (OPX+ 1/2, digital 1, Octave RF_outputs/1)
########################################################################################################################
_ao = machine.ports.analog_outputs.setdefault("con1", {})
_do = machine.ports.digital_outputs.setdefault("con1", {})

for port_id in (1, 2):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=False)

if 1 not in _do:
    _do[1] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=1, shareable=False)

rr_rf1 = machine.octaves["oct1"].RF_outputs[1]
rr_rf1.LO_frequency = rr_LO
rr_rf1.LO_source    = "internal"
rr_rf1.output_mode  = "triggered"
rr_rf1.gain = get_octave_gain_and_amplitude(readout_power, 0.125)[0]

# RF_inputs/2: down-converter shares LO with RF_outputs/1
rr_rfi2 = machine.octaves["oct1"].RF_inputs[2]
rr_rfi2.LO_source    = "internal"
rr_rfi2.LO_frequency = "#/octaves/oct1/RF_outputs/1/LO_frequency"

# OPX+ analog input ADC ports (con1, ports 1/2)
_ai = machine.ports.analog_inputs.setdefault("con1", {})
for port_id in (1, 2):
    if port_id not in _ai:
        _ai[port_id] = OPXPlusAnalogInputPort(
            controller_id="con1", port_id=port_id, gain_db=readout_adc_gain_db)
    else:
        _ai[port_id].gain_db = readout_adc_gain_db

########################################################################################################################
# %%  Qubit XY hardware (OPX+ 5/6, digital 5, Octave RF_outputs/3)
########################################################################################################################
for port_id in (5, 6):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=False)

if 5 not in _do:
    _do[5] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=5, shareable=False)

xy_rf3 = machine.octaves["oct1"].RF_outputs[3]
xy_rf3.LO_frequency = xy_LO
xy_rf3.LO_source    = "internal"
xy_rf3.output_mode  = "triggered"
if xy_rf3.gain is None:
    xy_rf3.gain = get_octave_gain_and_amplitude(drive_power)[0]

########################################################################################################################
# %%  f0g1 sideband drive hardware (OPX+ 3/4 I/Q, digital 3, Octave RF_outputs/2, ext LO)
########################################################################################################################
for port_id in (3, 4):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=True)
    else:
        _ao[port_id].shareable = True

if 3 not in _do:
    _do[3] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=3, shareable=False)

f0g1_rf2 = machine.octaves["oct1"].RF_outputs[2]
f0g1_rf2.LO_frequency = alice_f0g1_LO
f0g1_rf2.LO_source    = "external"  # RF2 uses an external LO connected to Octave LO2
f0g1_rf2.gain         = alice_f0g1_gain
f0g1_rf2.output_mode  = "triggered"

########################################################################################################################
# %%  Cavity hardware (OPX+ 7/8, digital 7, Octave RF_outputs/4) -- alice + bob share ports
########################################################################################################################
for port_id in (7, 8):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=True)
    else:
        _ao[port_id].shareable = True

if 7 not in _do:
    _do[7] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=7, shareable=True)
else:
    _do[7].shareable = True

cav_rf4 = machine.octaves["oct1"].RF_outputs[4]
cav_gain, cav_amp = get_octave_gain_and_amplitude(alice_power)
cav_rf4.LO_frequency = alice_LO
cav_rf4.LO_source    = "internal"
cav_rf4.gain         = cav_gain
cav_rf4.output_mode  = "triggered"

# RF5: unused -- keep always off
rf5 = machine.octaves["oct1"].RF_outputs[5]
rf5.LO_source   = "internal"
rf5.output_mode = "always_off"

# Loopbacks: cleared -- all channels use internal Octave LOs except f0g1
machine.octaves["oct1"].loopbacks = []

########################################################################################################################
# %%  Resonator -- frequencies and pulses
########################################################################################################################
rr_gain, rr_amp = get_octave_gain_and_amplitude(readout_power, 0.125)
for qubit in machine.qubits.values():
    qubit.resonator.f_01         = rr_freq
    qubit.resonator.RF_frequency = rr_freq
    qubit.resonator.frequency_converter_up.LO_frequency = rr_LO
    qubit.resonator.frequency_converter_up.gain         = rr_gain
    qubit.resonator.frequency_converter_up.output_mode  = "triggered"
    qubit.resonator.depletion_time = resonator_depletion_time_ns
    qubit.resonator.frequency_bare = rr_freq_bare
    qubit.resonator.time_of_flight = tof_ns
    qubit.resonator.smearing       = 0
    # f_12, GEF_frequency_shift, confusion_matrix, gef_centers, gef_confusion_matrix
    # stay None -- populated by readout calibration nodes (07, 08a, 14, 15)
    ro_op = qubit.resonator.operations.get("readout")
    if ro_op is not None:
        ro_op.amplitude = rr_amp
        ro_op.length    = readout_length_ns

########################################################################################################################
# %%  Qubit XY -- frequencies and pulses
########################################################################################################################
xy_gain, xy_amp = get_octave_gain_and_amplitude(drive_power)
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.f_01                                   = xy_freq
    qubit.xy.RF_frequency                        = xy_freq
    qubit.xy.frequency_converter_up.LO_frequency = xy_LO
    qubit.xy.frequency_converter_up.gain         = xy_gain
    qubit.xy.frequency_converter_up.output_mode  = "triggered"
    qubit.T1                                     = T1
    qubit.anharmonicity                          = int(anharmonicity)
    qubit.grid_location                          = f"{k},0"
    qubit.thermalization_time_factor             = thermalization_time_factor
    qubit.sigma_time_factor                      = sigma_time_factor
    # f_12, T2ramsey, T2echo, chi stay None -- calibrated by nodes 06a, 06b, 20
    sat_op = qubit.xy.operations.get("saturation")
    if sat_op is not None:
        sat_op.amplitude      = 0.3
        sat_op.length         = saturation_length_ns
        sat_op.digital_marker = "ON"
    add_DragGaussian_pulses(qubit, xy_amp, x180_length_ns, gaussian_sigma_ns,
                            drag_alpha, drag_detuning, anharmonicity,
                            digital_marker="ON")

########################################################################################################################
# %%  EF and selective pulses
########################################################################################################################
for q_name, qubit in machine.qubits.items():
    xy = qubit.xy
    xy.operations["EF_x180"] = DragGaussianPulse(
        length="#../x180/length",
        amplitude=ef_x180_amplitude,
        sigma="#../x180/sigma",
        alpha=0.0,
        anharmonicity=int(anharmonicity),
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    xy.operations["EF_x90"] = DragGaussianPulse(
        length="#../EF_x180/length",
        amplitude=ef_x180_amplitude / 2,
        sigma="#../EF_x180/sigma",
        alpha="#../EF_x180/alpha",
        anharmonicity="#../EF_x180/anharmonicity",
        detuning="#../EF_x180/detuning",
        subtracted="#../EF_x180/subtracted",
        axis_angle=0,
        digital_marker="#../EF_x180/digital_marker",
    )
    # Amplitude proportional to 1/length to preserve pulse area (same rotation angle as x180).
    sel_amp = xy_amp * (x180_length_ns / selective_x180_length_ns)
    xy.operations["selective_x180"] = DragGaussianPulse(
        length=selective_x180_length_ns,
        amplitude=sel_amp,
        sigma=selective_x180_length_ns // 5,
        alpha=0.0,
        anharmonicity=int(anharmonicity),
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )

########################################################################################################################
# %%  Cavity object (alice + bob)
########################################################################################################################
if CAVITY_ID not in machine.cavities:
    def _make_cavity_drive() -> XYDriveIQ:
        drive = XYDriveIQ(
            opx_output_I="#/ports/analog_outputs/con1/7",
            opx_output_Q="#/ports/analog_outputs/con1/8",
            frequency_converter_up="#/octaves/oct1/RF_outputs/4",
            RF_frequency=None,
        )
        drive.digital_outputs["octave_switch_0"] = DigitalOutputChannel(
            opx_output="#/ports/digital_outputs/con1/7", delay=57, buffer=18,
        )
        return drive

    alice_mode = CavityMode(id="alice", cavity_mode_drive=_make_cavity_drive())
    alice_mode.T1 = cavity_T1
    bob_mode = CavityMode(id="bob", cavity_mode_drive=_make_cavity_drive())
    bob_mode.T1 = cavity_T1
    machine.cavities[CAVITY_ID] = Cavity(id=CAVITY_ID, alice=alice_mode, bob=bob_mode)
    cav_rf4.channel = f"#/cavities/{CAVITY_ID}/alice/cavity_mode_drive"
    print(f"  Created cavity '{CAVITY_ID}' with alice and bob modes.")
else:
    for mode_name in ("alice", "bob"):
        mode = getattr(machine.cavities[CAVITY_ID], mode_name, None)
        if mode is not None:
            mode.T1 = cavity_T1

# Cavity drive frequencies and pulses -- always reset to initial values
for cav_name, cavity in machine.cavities.items():
    for mode_name, freq in (("alice", alice_freq), ("bob", bob_freq)):
        mode = getattr(cavity, mode_name, None)
        if mode is None or mode.cavity_mode_drive is None:
            continue
        mode.cavity_mode_drive.RF_frequency = freq
        mode.cavity_mode_drive.operations["saturation"] = SquarePulse(
            length=readout_length_ns, amplitude=cav_amp, digital_marker="ON")
        mode.cavity_mode_drive.operations["displacement"] = DragGaussianPulse(
            length=displacement_length_ns,
            amplitude=displacement_initial_amplitude_V,
            sigma=displacement_sigma_ns,
            alpha=0.0,
            anharmonicity=0,
            detuning=0.0,
            axis_angle=0,
            digital_marker="ON",
        )

# Thermalization factors: alice 3xT1, bob 5xT1
CAVITY_THERM_FACTORS = {"alice": 3, "bob": 5}
for cav_name, cavity in machine.cavities.items():
    for mode_name, factor in CAVITY_THERM_FACTORS.items():
        mode = getattr(cavity, mode_name, None)
        if mode is not None:
            mode.thermalization_time_factor = factor

########################################################################################################################
# %%  CavityTransmonPair (with f0g1 sideband_drive)
########################################################################################################################
for q_name in machine.qubits:
    for mode_name in ("alice", "bob"):
        pair_key = f"{q_name}_{mode_name}"
        if pair_key not in machine.cavity_transmon_pairs:
            machine.cavity_transmon_pairs[pair_key] = CavityTransmonPair(
                qubit_name=q_name, cavity_mode_name=mode_name
            )
            print(f"  Created CavityTransmonPair '{pair_key}'")

    # parity_time calibrated by node 30; initialise to None on first run
    for pk in (f"{q_name}_alice", f"{q_name}_bob"):
        p = machine.cavity_transmon_pairs.get(pk)
        if p is not None and not hasattr(p, "parity_time"):
            p.parity_time = None

    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is not None and alice_pair.sideband_drive is None:
        alice_drive = XYDriveIQ(
            id=f"{q_name}_alice_f0g1",
            opx_output_I="#/ports/analog_outputs/con1/3",
            opx_output_Q="#/ports/analog_outputs/con1/4",
            frequency_converter_up="#/octaves/oct1/RF_outputs/2",
            RF_frequency=alice_f0g1_freq,
        )
        alice_drive.digital_outputs["octave_switch_0"] = DigitalOutputChannel(
            opx_output="#/ports/digital_outputs/con1/3", delay=57, buffer=18,
        )
        alice_drive.operations["saturation"] = SquarePulse(
            length=alice_f0g1_saturation_length_ns,
            amplitude=alice_f0g1_amp,
            digital_marker="ON",
        )
        alice_drive.operations["f0g1_pi"] = DragCosinePulse(
            length=alice_f0g1_pi_length_ns,
            axis_angle=0.0,
            alpha=0.0,
            anharmonicity=0,
            amplitude=alice_f0g1_amp,
            digital_marker="ON",
        )
        alice_pair.sideband_drive = alice_drive
        f0g1_rf2.channel = f"#/cavity_transmon_pairs/{q_name}_alice/sideband_drive"
        print(f"  Created sideband_drive for '{q_name}_alice' on RF_outputs/2.")
    elif alice_pair is not None and alice_pair.sideband_drive is not None:
        alice_pair.sideband_drive.RF_frequency = alice_f0g1_freq
    # Note: bob sideband_drive stays None -- requires separate hardware

########################################################################################################################
# %%  Temporary calibration state
########################################################################################################################
if machine.temp_calibration is None:
    machine.temp_calibration = {}
for q_name in machine.qubits:
    if q_name not in machine.temp_calibration:
        machine.temp_calibration[q_name] = TemporaryCalibrationData(
            initial_resonator_f01=rr_freq,
            initial_resonator_RF_frequency=rr_freq,
        )
        print(f"  Initialized temp_calibration['{q_name}']")
    else:
        tc = machine.temp_calibration[q_name]
        tc.initial_resonator_f01          = rr_freq
        tc.initial_resonator_RF_frequency = rr_freq

########################################################################################################################
# %%  Save
########################################################################################################################
machine.save()
print("QUAM saved.")
with open("qua_config.json", "w+") as f:
    json.dump(machine.generate_config(), f, indent=4)
print("QUA config saved.")

## 3. Verify state

Sanity-check the saved QUAM structure after running sections 1 and 2.

In [ ]:
from quam_config import Quam

machine = Quam.load()

print("=== Qubits ===")
for q_name, qubit in machine.qubits.items():
    print(f"  {q_name}: f_01={qubit.f_01/1e9:.4f} GHz, T1={qubit.T1*1e6:.0f} us")
    rr = qubit.resonator
    print(f"    resonator: f={rr.f_01/1e9:.4f} GHz, LO={rr.frequency_converter_up.LO_frequency/1e9:.4f} GHz, "
          f"gain={rr.frequency_converter_up.gain} dB")
    xy = qubit.xy
    print(f"    xy:        f={xy.RF_frequency/1e9:.4f} GHz, LO={xy.frequency_converter_up.LO_frequency/1e9:.4f} GHz, "
          f"gain={xy.frequency_converter_up.gain} dB")

print("\n=== Octave RF outputs ===")
for rf_id, rf_out in machine.octaves["oct1"].RF_outputs.items():
    print(f"  RF_outputs/{rf_id}: LO={rf_out.LO_frequency/1e9:.4f} GHz, "
          f"source={rf_out.LO_source}, gain={rf_out.gain} dB, mode={rf_out.output_mode}")

print("\n=== Cavities ===")
for cav_name, cavity in machine.cavities.items():
    print(f"  {cav_name}:")
    for mode_name in ("alice", "bob"):
        mode = getattr(cavity, mode_name, None)
        if mode is not None and mode.cavity_mode_drive is not None:
            print(f"    {mode_name}: RF={mode.cavity_mode_drive.RF_frequency/1e9:.4f} GHz, T1={mode.T1*1e6:.0f} us")

print("\n=== CavityTransmonPairs ===")
for pair_key, pair in machine.cavity_transmon_pairs.items():
    sb = pair.sideband_drive
    sb_info = f"sideband_drive RF={sb.RF_frequency/1e9:.4f} GHz" if sb is not None else "no sideband_drive"
    print(f"  {pair_key}: {sb_info}")